# 14 — Market Basket Analysis

## Objective

This notebook examines which product categories appear together within the same order using transparent association metrics:

- **Support:** Share of orders containing an item or category pair
- **Confidence:** Share of orders containing category A that also contain category B
- **Lift:** Pair co-occurrence relative to what would be expected if the categories appeared independently

The analysis identifies historical co-purchase patterns. It does not prove that one category causes another purchase or quantify the impact of a future recommendation strategy.

## 1. Data Loading

Load the cleaned order-item, product, and category-translation tables required to construct category-level customer baskets.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from itertools import combinations
from collections import Counter
from matplotlib.ticker import PercentFormatter

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.4f}".format)

DATA_DIR = Path("../data/processed")
IMAGES_DIR = Path("../images/notebook_outputs")

IMAGES_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {
    "primary": "#173F5F",
    "secondary": "#20639B",
    "accent": "#F6C85F",
    "positive": "#3CAEA3",
    "negative": "#ED553B",
    "neutral": "#8A8A8A"
}

order_items = pd.read_csv(
    DATA_DIR / "order_items_clean.csv"
)

products = pd.read_csv(
    DATA_DIR / "products_clean.csv"
)

category_translation = pd.read_csv(
    DATA_DIR / "category_translation_clean.csv"
)

dataset_summary = pd.DataFrame({
    "dataset": [
        "order_items",
        "products",
        "category_translation"
    ],
    "rows": [
        len(order_items),
        len(products),
        len(category_translation)
    ],
    "columns": [
        order_items.shape[1],
        products.shape[1],
        category_translation.shape[1]
    ]
})

display(
    dataset_summary.style.format({
        "rows": "{:,.0f}",
        "columns": "{:,.0f}"
    })
)

print(
    "Orders represented:",
    f"{order_items['order_id'].nunique():,}"
)
print(
    "Products represented:",
    f"{order_items['product_id'].nunique():,}"
)

,dataset,rows,columns
0,order_items,"112,650",7
1,products,"32,951",10
2,category_translation,74,2


Orders represented: 98,666
Products represented: 32,951


## 2. Category Integration and Validation

Join each order item to its cleaned English product category and confirm that the integration preserves item counts and financial totals without creating duplicate rows.

In [2]:
item_categories = (
    order_items
    .merge(
        products[
            [
                "product_id",
                "product_category_name"
            ]
        ],
        on="product_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        category_translation,
        on="product_category_name",
        how="left",
        validate="many_to_one"
    )
)

category_validation = pd.DataFrame([
    {
        "validation_check": "Item rows retained",
        "result": len(item_categories),
        "expected": len(order_items)
    },
    {
        "validation_check": "Missing product matches",
        "result": item_categories[
            "product_category_name"
        ].isna().sum(),
        "expected": 0
    },
    {
        "validation_check": "Missing English categories",
        "result": item_categories[
            "product_category_name_english"
        ].isna().sum(),
        "expected": 0
    },
    {
        "validation_check": "Product value retained",
        "result": round(
            item_categories["price"].sum(),
            2
        ),
        "expected": round(
            order_items["price"].sum(),
            2
        )
    },
    {
        "validation_check": "Freight value retained",
        "result": round(
            item_categories[
                "freight_value"
            ].sum(),
            2
        ),
        "expected": round(
            order_items[
                "freight_value"
            ].sum(),
            2
        )
    }
])

category_validation["status"] = np.where(
    category_validation["result"]
    == category_validation["expected"],
    "Passed",
    "Failed"
)

display(category_validation)

print(
    "English categories represented:",
    item_categories[
        "product_category_name_english"
    ].nunique()
)

,validation_check,result,expected,status
0,Item rows retained,"112,650.0000","112,650.0000",Passed
1,Missing product matches,0.0000,0.0000,Passed
2,Missing English categories,0.0000,0.0000,Passed
3,Product value retained,"13,591,643.7000","13,591,643.7000",Passed
4,Freight value retained,"2,251,909.5400","2,251,909.5400",Passed


English categories represented: 74


## 3. Basket Structure Assessment

Measure item, product, and category counts per order to determine how much of the dataset is suitable for category-level market basket analysis.

In [3]:
basket_profile = (
    item_categories
    .groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "size"),
        distinct_products=("product_id", "nunique"),
        distinct_categories=(
            "product_category_name_english",
            "nunique"
        ),
        product_value=("price", "sum")
    )
)

order_baskets = (
    item_categories
    .groupby("order_id")[
        "product_category_name_english"
    ]
    .agg(
        lambda categories:
            tuple(sorted(set(categories)))
    )
    .rename("category_basket")
    .reset_index()
)

basket_profile = (
    basket_profile
    .merge(
        order_baskets,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

basket_summary = pd.DataFrame({
    "metric": [
        "Orders with item records",
        "Orders with multiple item records",
        "Orders with multiple distinct products",
        "Orders with multiple distinct categories",
        "Multi-category order rate",
        "Maximum items in one order",
        "Maximum distinct products in one order",
        "Maximum distinct categories in one order"
    ],
    "value": [
        len(basket_profile),
        (basket_profile["item_count"] > 1).sum(),
        (
            basket_profile[
                "distinct_products"
            ] > 1
        ).sum(),
        (
            basket_profile[
                "distinct_categories"
            ] > 1
        ).sum(),
        (
            basket_profile[
                "distinct_categories"
            ] > 1
        ).mean(),
        basket_profile["item_count"].max(),
        basket_profile["distinct_products"].max(),
        basket_profile[
            "distinct_categories"
        ].max()
    ]
})

display(
    basket_summary.style.format({
        "value": lambda value: (
            f"{value:.2%}"
            if isinstance(value, (float, np.floating))
            and value < 1
            else f"{value:,.0f}"
        )
    })
)

category_count_distribution = (
    basket_profile[
        "distinct_categories"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("distinct_categories")
    .reset_index(name="orders")
)

category_count_distribution["order_share"] = (
    category_count_distribution["orders"]
    / category_count_distribution["orders"].sum()
)

display(
    category_count_distribution.style.format({
        "orders": "{:,.0f}",
        "order_share": "{:.2%}"
    })
)

,metric,value
0,Orders with item records,"98,666"
1,Orders with multiple item records,"9,803"
2,Orders with multiple distinct products,"3,236"
3,Orders with multiple distinct categories,786
4,Multi-category order rate,0.80%
5,Maximum items in one order,21
6,Maximum distinct products in one order,8
7,Maximum distinct categories in one order,3


,distinct_categories,orders,order_share
0,1,"97,880",99.20%
1,2,768,0.78%
2,3,18,0.02%


## 4. Category Prevalence

Calculate how frequently each category appears across orders, providing the individual-category support required for confidence and lift calculations.

In [4]:
total_orders = (
    basket_profile["order_id"].nunique()
)

category_prevalence = (
    item_categories
    .groupby(
        "product_category_name_english",
        as_index=False
    )
    .agg(
        orders_containing_category=(
            "order_id",
            "nunique"
        ),
        item_records=("order_item_id", "size"),
        product_value=("price", "sum")
    )
)

category_prevalence["category_support"] = (
    category_prevalence[
        "orders_containing_category"
    ]
    / total_orders
)

category_prevalence["product_value_share"] = (
    category_prevalence["product_value"]
    / category_prevalence[
        "product_value"
    ].sum()
)

category_prevalence = (
    category_prevalence
    .sort_values(
        "orders_containing_category",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    category_prevalence
    .head(15)
    .style.format({
        "orders_containing_category": "{:,.0f}",
        "item_records": "{:,.0f}",
        "product_value": "R$ {:,.2f}",
        "category_support": "{:.2%}",
        "product_value_share": "{:.2%}"
    })
)

print(
    "Categories appearing in at least 100 orders:",
    f"{(category_prevalence['orders_containing_category'] >= 100).sum():,}"
)

,product_category_name_english,orders_containing_category,item_records,product_value,category_support,product_value_share
0,bed_bath_table,"9,417","11,115","R$ 1,036,988.68",9.54%,7.63%
1,health_beauty,"8,836","9,670","R$ 1,258,681.34",8.96%,9.26%
2,sports_leisure,"7,720","8,641","R$ 988,048.97",7.82%,7.27%
3,computers_accessories,"6,689","7,827","R$ 911,954.32",6.78%,6.71%
4,furniture_decor,"6,449","8,334","R$ 729,762.49",6.54%,5.37%
5,housewares,"5,884","6,964","R$ 632,248.66",5.96%,4.65%
6,watches_gifts,"5,624","5,991","R$ 1,205,005.68",5.70%,8.87%
7,telephony,"4,199","4,545","R$ 323,667.53",4.26%,2.38%
8,auto,"3,897","4,235","R$ 592,720.11",3.95%,4.36%
9,toys,"3,886","4,117","R$ 483,946.60",3.94%,3.56%


Categories appearing in at least 100 orders: 52


## 5. Category-Pair Association Metrics

Generate unique category pairs from each order and calculate pair count, support, directional confidence, and lift using all orders with item records as the denominator.

In [5]:
pair_counter = Counter()

for basket in basket_profile["category_basket"]:
    if len(basket) >= 2:
        pair_counter.update(
            combinations(basket, 2)
        )

category_pairs = pd.DataFrame([
    {
        "category_a": category_a,
        "category_b": category_b,
        "pair_order_count": pair_count
    }
    for (
        category_a,
        category_b
    ), pair_count in pair_counter.items()
])

category_order_lookup = (
    category_prevalence
    .set_index(
        "product_category_name_english"
    )["orders_containing_category"]
)

category_pairs["category_a_orders"] = (
    category_pairs["category_a"]
    .map(category_order_lookup)
)

category_pairs["category_b_orders"] = (
    category_pairs["category_b"]
    .map(category_order_lookup)
)

category_pairs["pair_support"] = (
    category_pairs["pair_order_count"]
    / total_orders
)

category_pairs["category_a_support"] = (
    category_pairs["category_a_orders"]
    / total_orders
)

category_pairs["category_b_support"] = (
    category_pairs["category_b_orders"]
    / total_orders
)

category_pairs["confidence_a_to_b"] = (
    category_pairs["pair_order_count"]
    / category_pairs["category_a_orders"]
)

category_pairs["confidence_b_to_a"] = (
    category_pairs["pair_order_count"]
    / category_pairs["category_b_orders"]
)

category_pairs["lift"] = (
    category_pairs["pair_support"]
    / (
        category_pairs["category_a_support"]
        * category_pairs["category_b_support"]
    )
)

expected_pair_occurrences = int(
    basket_profile["distinct_categories"]
    .apply(
        lambda category_count:
            category_count
            * (category_count - 1)
            / 2
    )
    .sum()
)

pair_validation = pd.DataFrame([
    {
        "validation_check":
            "Category-pair occurrences represented",
        "result": category_pairs[
            "pair_order_count"
        ].sum(),
        "expected": expected_pair_occurrences
    },
    {
        "validation_check":
            "Duplicate unordered category pairs",
        "result": category_pairs[
            ["category_a", "category_b"]
        ].duplicated().sum(),
        "expected": 0
    },
    {
        "validation_check":
            "Invalid pair supports",
        "result": (
            ~category_pairs[
                "pair_support"
            ].between(0, 1)
        ).sum(),
        "expected": 0
    }
])

pair_validation["status"] = np.where(
    pair_validation["result"]
    == pair_validation["expected"],
    "Passed",
    "Failed"
)

display(pair_validation)

minimum_category_orders = 100
minimum_pair_orders = 5

reliable_category_pairs = (
    category_pairs.loc[
        (
            category_pairs["category_a_orders"]
            >= minimum_category_orders
        )
        & (
            category_pairs["category_b_orders"]
            >= minimum_category_orders
        )
        & (
            category_pairs["pair_order_count"]
            >= minimum_pair_orders
        )
    ]
    .sort_values(
        [
            "pair_order_count",
            "lift"
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

display(
    reliable_category_pairs
    .head(15)
    .style.format({
        "pair_order_count": "{:,.0f}",
        "category_a_orders": "{:,.0f}",
        "category_b_orders": "{:,.0f}",
        "pair_support": "{:.4%}",
        "category_a_support": "{:.2%}",
        "category_b_support": "{:.2%}",
        "confidence_a_to_b": "{:.2%}",
        "confidence_b_to_a": "{:.2%}",
        "lift": "{:.2f}"
    })
)

print(
    "All observed category pairs:",
    f"{len(category_pairs):,}"
)
print(
    "Pairs meeting reliability filters:",
    f"{len(reliable_category_pairs):,}"
)

,validation_check,result,expected,status
0,Category-pair occurrences represented,822,822,Passed
1,Duplicate unordered category pairs,0,0,Passed
2,Invalid pair supports,0,0,Passed


,category_a,category_b,pair_order_count,category_a_orders,category_b_orders,pair_support,category_a_support,category_b_support,confidence_a_to_b,confidence_b_to_a,lift
0,bed_bath_table,furniture_decor,70,"9,417","6,449",0.0709%,9.54%,6.54%,0.74%,1.09%,0.11
1,bed_bath_table,home_confort,43,"9,417",397,0.0436%,9.54%,0.40%,0.46%,10.83%,1.13
2,furniture_decor,housewares,24,"6,449","5,884",0.0243%,6.54%,5.96%,0.37%,0.41%,0.06
3,baby,cool_stuff,20,"2,885","3,632",0.0203%,2.92%,3.68%,0.69%,0.55%,0.19
4,bed_bath_table,housewares,20,"9,417","5,884",0.0203%,9.54%,5.96%,0.21%,0.34%,0.04
5,baby,toys,19,"2,885","3,886",0.0193%,2.92%,3.94%,0.66%,0.49%,0.17
6,furniture_decor,garden_tools,17,"6,449","3,518",0.0172%,6.54%,3.57%,0.26%,0.48%,0.07
7,baby,bed_bath_table,17,"2,885","9,417",0.0172%,2.92%,9.54%,0.59%,0.18%,0.06
8,housewares,unknown,14,"5,884","1,451",0.0142%,5.96%,1.47%,0.24%,0.96%,0.16
9,health_beauty,sports_leisure,14,"8,836","7,720",0.0142%,8.96%,7.82%,0.16%,0.18%,0.02


All observed category pairs: 262
Pairs meeting reliability filters: 41


***Observation:*** The dataset contains only 822 category-pair occurrences across 98,666 orders. The most frequent pair appears in 70 orders with support of 0.0709%, while many high-count pairs have lift below 1, meaning they co-occur less often than expected from their individual prevalence. Pair frequency alone is therefore not evidence of a strong association.

## 6. Directional Association Rules

Convert filtered category pairs into directional rules and compare confidence with lift, retaining the low support and limited pair counts required for cautious interpretation.

In [6]:
filtered_category_pairs = (
    reliable_category_pairs.copy()
)

rules_a_to_b = pd.DataFrame({
    "antecedent":
        filtered_category_pairs["category_a"],
    "consequent":
        filtered_category_pairs["category_b"],
    "pair_order_count":
        filtered_category_pairs["pair_order_count"],
    "antecedent_orders":
        filtered_category_pairs["category_a_orders"],
    "consequent_orders":
        filtered_category_pairs["category_b_orders"],
    "support":
        filtered_category_pairs["pair_support"],
    "confidence":
        filtered_category_pairs["confidence_a_to_b"],
    "lift":
        filtered_category_pairs["lift"]
})

rules_b_to_a = pd.DataFrame({
    "antecedent":
        filtered_category_pairs["category_b"],
    "consequent":
        filtered_category_pairs["category_a"],
    "pair_order_count":
        filtered_category_pairs["pair_order_count"],
    "antecedent_orders":
        filtered_category_pairs["category_b_orders"],
    "consequent_orders":
        filtered_category_pairs["category_a_orders"],
    "support":
        filtered_category_pairs["pair_support"],
    "confidence":
        filtered_category_pairs["confidence_b_to_a"],
    "lift":
        filtered_category_pairs["lift"]
})

association_rules = (
    pd.concat(
        [
            rules_a_to_b,
            rules_b_to_a
        ],
        ignore_index=True
    )
    .sort_values(
        [
            "lift",
            "confidence",
            "pair_order_count"
        ],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

positive_lift_rules = (
    association_rules.loc[
        association_rules["lift"] > 1
    ]
    .reset_index(drop=True)
)

display(
    positive_lift_rules
    .head(20)
    .style.format({
        "pair_order_count": "{:,.0f}",
        "antecedent_orders": "{:,.0f}",
        "consequent_orders": "{:,.0f}",
        "support": "{:.4%}",
        "confidence": "{:.2%}",
        "lift": "{:.2f}"
    })
)

print(
    "Directional rules after filtering:",
    f"{len(association_rules):,}"
)
print(
    "Directional rules with lift above 1:",
    f"{len(positive_lift_rules):,}"
)
print(
    "Maximum observed lift:",
    f"{association_rules['lift'].max():.2f}"
)

,antecedent,consequent,pair_order_count,antecedent_orders,consequent_orders,support,confidence,lift
0,home_confort,bed_bath_table,43,397,"9,417",0.0436%,10.83%,1.13
1,bed_bath_table,home_confort,43,"9,417",397,0.0436%,0.46%,1.13


Directional rules after filtering: 82
Directional rules with lift above 1: 2
Maximum observed lift: 1.13


## 7. Association-Rule Threshold Sensitivity

Test how the number and strength of retained category pairs changes under alternative category-prevalence and pair-count thresholds, reducing the risk of selecting one favorable filter combination.

In [7]:
category_order_thresholds = [
    50,
    100,
    500
]

pair_order_thresholds = [
    3,
    5,
    10
]

sensitivity_records = []

for category_threshold in (
    category_order_thresholds
):
    for pair_threshold in (
        pair_order_thresholds
    ):
        filtered_pairs = (
            category_pairs.loc[
                (
                    category_pairs[
                        "category_a_orders"
                    ] >= category_threshold
                )
                & (
                    category_pairs[
                        "category_b_orders"
                    ] >= category_threshold
                )
                & (
                    category_pairs[
                        "pair_order_count"
                    ] >= pair_threshold
                )
            ]
        )

        positive_pairs = (
            filtered_pairs.loc[
                filtered_pairs["lift"] > 1
            ]
        )

        sensitivity_records.append({
            "minimum_category_orders":
                category_threshold,
            "minimum_pair_orders":
                pair_threshold,
            "retained_unordered_pairs":
                len(filtered_pairs),
            "pairs_with_lift_above_1":
                len(positive_pairs),
            "maximum_lift": (
                filtered_pairs["lift"].max()
                if len(filtered_pairs) > 0
                else np.nan
            ),
            "maximum_pair_support": (
                filtered_pairs[
                    "pair_support"
                ].max()
                if len(filtered_pairs) > 0
                else np.nan
            )
        })

threshold_sensitivity = pd.DataFrame(
    sensitivity_records
)

display(
    threshold_sensitivity.style.format({
        "minimum_category_orders": "{:,.0f}",
        "minimum_pair_orders": "{:,.0f}",
        "retained_unordered_pairs": "{:,.0f}",
        "pairs_with_lift_above_1": "{:,.0f}",
        "maximum_lift": "{:.2f}",
        "maximum_pair_support": "{:.4%}"
    })
)

print(
    "Threshold combinations tested:",
    f"{len(threshold_sensitivity):,}"
)

,minimum_category_orders,minimum_pair_orders,retained_unordered_pairs,pairs_with_lift_above_1,maximum_lift,maximum_pair_support
0,50,3,67,2,1.40,0.0709%
1,50,5,41,1,1.13,0.0709%
2,50,10,18,1,1.13,0.0709%
3,100,3,67,2,1.40,0.0709%
4,100,5,41,1,1.13,0.0709%
5,100,10,18,1,1.13,0.0709%
6,500,3,53,0,0.33,0.0709%
7,500,5,33,0,0.33,0.0709%
8,500,10,15,0,0.19,0.0709%


Threshold combinations tested: 9


***Observation:*** The results are sensitive to the selected thresholds. Only one or two positive-lift category pairs remain when categories with at least 50–100 orders are included, while none remain at the 500-order threshold. Since maximum pair support is only 0.0709%, these associations are too sparse for dependable cross-selling recommendations and should be treated as exploratory.

## 8. Final Validation

Confirm that the market-basket tables preserve the source order and category information and that all calculated association metrics remain within valid ranges.

In [8]:
expected_pair_occurrences = basket_profile[
    "distinct_categories"
].apply(
    lambda category_count: (
        category_count * (category_count - 1)
    ) // 2
).sum()

invalid_pair_metrics = (
    category_pairs[
        [
            "pair_support",
            "category_a_support",
            "category_b_support",
            "confidence_a_to_b",
            "confidence_b_to_a"
        ]
    ]
    .lt(0)
    .any(axis=1)
    |
    category_pairs[
        [
            "pair_support",
            "category_a_support",
            "category_b_support",
            "confidence_a_to_b",
            "confidence_b_to_a"
        ]
    ]
    .gt(1)
    .any(axis=1)
).sum()

invalid_lift_values = (
    category_pairs["lift"].isna()
    | category_pairs["lift"].le(0)
).sum()

duplicate_directional_rules = association_rules.duplicated(
    subset=["antecedent", "consequent"]
).sum()

market_basket_validation = pd.DataFrame({
    "validation_check": [
        "Orders represented in basket profiles",
        "Duplicate basket-profile order IDs",
        "Category-pair occurrences represented",
        "Duplicate unordered category pairs",
        "Filtered unordered pairs represented",
        "Directional rules represented",
        "Duplicate directional rules",
        "Invalid support or confidence values",
        "Invalid lift values",
        "Threshold combinations represented"
    ],
    "result": [
        len(basket_profile),
        basket_profile["order_id"].duplicated().sum(),
        category_pairs["pair_order_count"].sum(),
        category_pairs.duplicated(
            subset=["category_a", "category_b"]
        ).sum(),
        len(filtered_category_pairs),
        len(association_rules),
        duplicate_directional_rules,
        invalid_pair_metrics,
        invalid_lift_values,
        len(threshold_sensitivity)
    ],
    "expected": [
        item_categories["order_id"].nunique(),
        0,
        expected_pair_occurrences,
        0,
        41,
        len(filtered_category_pairs) * 2,
        0,
        0,
        0,
        9
    ]
})

market_basket_validation["status"] = np.where(
    market_basket_validation["result"]
    == market_basket_validation["expected"],
    "Passed",
    "Failed"
)

display(
    market_basket_validation.style.format({
        "result": "{:,.0f}",
        "expected": "{:,.0f}"
    })
)

,validation_check,result,expected,status
0,Orders represented in basket profiles,"98,666","98,666",Passed
1,Duplicate basket-profile order IDs,0,0,Passed
2,Category-pair occurrences represented,822,822,Passed
3,Duplicate unordered category pairs,0,0,Passed
4,Filtered unordered pairs represented,41,41,Passed
5,Directional rules represented,82,82,Passed
6,Duplicate directional rules,0,0,Passed
7,Invalid support or confidence values,0,0,Passed
8,Invalid lift values,0,0,Passed
9,Threshold combinations represented,9,9,Passed


***Observation:*** All validation checks passed. The analysis represents 98,666 baskets and all 822 category-pair occurrences without duplicate combinations or invalid association metrics, while the filtered outputs retain the expected 41 unordered pairs and 82 directional rules.

## 9. Export Market-Basket Analysis Tables

Export the basket profiles, category summaries, association rules, sensitivity results, and validation report for reuse in reporting and portfolio documentation.

In [9]:
market_basket_export_files = {
    "market_basket_order_profiles.csv":
        basket_profile,

    "market_basket_category_prevalence.csv":
        category_prevalence,

    "market_basket_category_pairs.csv":
        category_pairs,

    "market_basket_filtered_category_pairs.csv":
        filtered_category_pairs,

    "market_basket_directional_rules.csv":
        association_rules,

    "market_basket_threshold_sensitivity.csv":
        threshold_sensitivity,

    "market_basket_validation.csv":
        market_basket_validation
}

for filename, dataframe in market_basket_export_files.items():
    dataframe.to_csv(
        DATA_DIR / filename,
        index=False
    )

all_market_basket_files_created = all(
    (DATA_DIR / filename).exists()
    for filename in market_basket_export_files
)

print(
    "Market-basket files exported:",
    len(market_basket_export_files)
)

print(
    "All market-basket files created:",
    all_market_basket_files_created
)

Market-basket files exported: 7
All market-basket files created: True


***Observation:*** All seven market-basket analysis files were exported successfully. The outputs preserve the complete analysis workflow, from order-level basket construction through association rules, threshold sensitivity, and final validation.

## 10. Conclusion

***Conclusion:*** The analysis found that cross-category purchasing is uncommon in the Olist data: only 786 of 98,666 orders (0.80%) contain more than one product category. Across all baskets, 822 category-pair occurrences produced 262 unique pairs, but the most frequent pair appeared in only 70 orders, representing 0.0709% of all analyzed baskets.

After applying minimum-prevalence and pair-frequency filters, 41 unordered category pairs and 82 directional rules remained. Only one unordered pair—`home_confort` and `bed_bath_table`—had lift above 1 under the primary thresholds, with a modest lift of 1.13 and support of only 0.0436%.

The threshold-sensitivity analysis showed that positive-lift associations disappear when the minimum category prevalence is increased to 500 orders. Therefore, the results demonstrate the market-basket analysis method, but the observed associations are too sparse and threshold-sensitive to support dependable cross-selling recommendations without additional transaction data or analysis at a more detailed product level.